In [ ]:
from datasets import load_dataset
from python_code_tokenizer import PythonCodeTokenizer
import torch
from torch.utils.data import Dataset, DataLoader

ModuleNotFoundError: No module named 'datasets'

In [3]:
# Load the Python code dataset
ds = load_dataset("flytech/python-codes-25k")
print(f"Dataset size: {len(ds['train'])}")
print("Sample code:")
print(ds['train'][0]['code'][:200])

NameError: name 'load_dataset' is not defined

In [ ]:
# Build tokenizer vocabulary from sample codes
sample_codes = [ds['train'][i]['code'] for i in range(min(1000, len(ds['train'])))]

tokenizer = PythonCodeTokenizer()
tokenizer.build_vocab(sample_codes)

print(f"Vocabulary size: {len(tokenizer.str_to_int)}")
print("Special tokens:", list(tokenizer.special_tokens.keys()))

In [ ]:
# Test tokenization
test_code = ds['train'][0]['code']
print("Original code:")
print(test_code[:300])

# Encode
ids = tokenizer.encode(test_code)
print(f"\nTokenized to {len(ids)} tokens")
print("First 20 token IDs:", ids[:20])

# Decode
decoded = tokenizer.decode(ids)
print("\nDecoded code:")
print(decoded[:300])

In [ ]:
class CodeDataset(Dataset):
    def __init__(self, codes, tokenizer, max_length=512):
        self.input_ids = []
        self.target_ids = []
        
        for code in codes:
            token_ids = tokenizer.encode(code)
            
            # Create input-target pairs for next token prediction
            if len(token_ids) > max_length:
                token_ids = token_ids[:max_length]
            
            if len(token_ids) > 1:
                input_chunk = token_ids[:-1]
                target_chunk = token_ids[1:]
                
                # Pad if necessary
                while len(input_chunk) < max_length - 1:
                    input_chunk.append(tokenizer.str_to_int["<|unk|>"])
                    target_chunk.append(tokenizer.str_to_int["<|unk|>"])
                
                self.input_ids.append(torch.tensor(input_chunk))
                self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
# Create dataset and dataloader
train_codes = [ds['train'][i]['code'] for i in range(100)]  # Use first 100 for demo
dataset = CodeDataset(train_codes, tokenizer, max_length=256)

dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Test the dataloader
for batch_idx, (inputs, targets) in enumerate(dataloader):
    print(f"Batch {batch_idx}:")
    print(f"Input shape: {inputs.shape}")
    print(f"Target shape: {targets.shape}")
    
    # Show first sample in batch
    sample_input = inputs[0][:20].tolist()
    sample_target = targets[0][:20].tolist()
    
    print("Sample input tokens:", [tokenizer.int_to_str[id] for id in sample_input])
    print("Sample target tokens:", [tokenizer.int_to_str[id] for id in sample_target])
    break